In [2]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
import pyarrow as pa, gc
import sys
import time

import os
import warnings
import math
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder


import xgboost as xgb
from lightgbm import LGBMRanker, early_stopping, log_evaluation
from catboost import CatBoostRanker, Pool


import torch
from torch.utils.data import Dataset

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
trainPath = Path("/kaggle/input/aeroclub-recsys-2025/train.parquet")
testPath = Path("/kaggle/input/aeroclub-recsys-2025/test.parquet")

# Now Create a class that process a dataframe

In [4]:
class PLDFPorcessor(): 
    def __init__(self, dfPath): 
        self.trainDFPl = pl.read_parquet(dfPath)
        self.trainDFPl = self.trainDFPl.drop([
            "Id",
            #"companyID",
            "profileId",
            "requestDate",
            #"nationality",
            "__index_level_0__",
            "miniRules1_percentage",
            "miniRules0_percentage",
            "legs1_segments3_operatingCarrier_code",
            "legs1_segments2_operatingCarrier_code",
            "legs1_segments1_operatingCarrier_code",
            "legs1_segments0_operatingCarrier_code",
            "legs0_segments3_operatingCarrier_code",
            "legs0_segments2_operatingCarrier_code",
            "legs0_segments1_operatingCarrier_code",
            "legs0_segments0_operatingCarrier_code",
            # dropping segment3 and segment2 columns as before
            "legs1_segments3_seatsAvailable",
            "legs1_segments3_flightNumber",
            "legs1_segments3_duration",
            "legs1_segments3_departureFrom_airport_iata",
            "legs1_segments3_cabinClass",
            "legs1_segments3_baggageAllowance_weightMeasurementType",
            "legs1_segments3_baggageAllowance_quantity",
            "legs1_segments3_arrivalTo_airport_iata",
            "legs1_segments3_arrivalTo_airport_city_iata",
            "legs1_segments2_seatsAvailable",
            "legs1_segments2_flightNumber",
            "legs1_segments2_duration",
            "legs1_segments2_departureFrom_airport_iata",
            "legs1_segments2_cabinClass",
            "legs1_segments2_baggageAllowance_weightMeasurementType",
            "legs1_segments2_baggageAllowance_quantity",
            "legs1_segments2_arrivalTo_airport_iata",
            "legs1_segments2_arrivalTo_airport_city_iata",
            #"legs1_segments1_seatsAvailable",
            "legs1_segments1_flightNumber",
            "legs1_segments1_duration",
            "legs1_segments1_departureFrom_airport_iata",
            #"legs1_segments1_cabinClass",
            #"legs1_segments1_baggageAllowance_weightMeasurementType",
            #"legs1_segments1_baggageAllowance_quantity",
            "legs1_segments1_arrivalTo_airport_iata",
            "legs1_segments1_arrivalTo_airport_city_iata",
            "legs0_segments3_seatsAvailable",
            "legs0_segments3_flightNumber",
            "legs0_segments3_duration",
            "legs0_segments3_departureFrom_airport_iata",
            "legs0_segments3_cabinClass",
            "legs0_segments3_baggageAllowance_weightMeasurementType",
            "legs0_segments3_baggageAllowance_quantity",
            "legs0_segments3_arrivalTo_airport_iata",
            "legs0_segments3_arrivalTo_airport_city_iata",
            "legs0_segments2_seatsAvailable",
            "legs0_segments2_flightNumber",
            "legs0_segments2_duration",
            "legs0_segments2_departureFrom_airport_iata",
            "legs0_segments2_cabinClass",
            "legs0_segments2_baggageAllowance_weightMeasurementType",
            "legs0_segments2_baggageAllowance_quantity",
            "legs0_segments2_arrivalTo_airport_iata",
            "legs0_segments2_arrivalTo_airport_city_iata",
            #"legs0_segments1_seatsAvailable",
            "legs0_segments1_flightNumber",
            "legs0_segments1_duration",
            "legs0_segments1_departureFrom_airport_iata",
            #"legs0_segments1_cabinClass",
            #"legs0_segments1_baggageAllowance_weightMeasurementType",
            #"legs0_segments1_baggageAllowance_quantity",
            "legs0_segments1_arrivalTo_airport_iata",
            "legs0_segments1_arrivalTo_airport_city_iata",
            # "corporateTariffCode",
            # These low-level iata columns dropped (if you want, keep for geo features)
            "legs0_segments0_arrivalTo_airport_city_iata", 
            "legs0_segments0_arrivalTo_airport_iata", 
            "legs0_segments0_departureFrom_airport_iata", 
            "legs0_segments0_duration", 
            "legs0_segments0_flightNumber", 
            "legs1_segments0_arrivalTo_airport_city_iata", 
            "legs1_segments0_arrivalTo_airport_iata", 
            "legs1_segments0_departureFrom_airport_iata", 
            "legs1_segments0_duration", 
            "legs1_segments0_flightNumber",
            # ADD THESE 5 TO REDUCE MEMORY:
            #"legs0_arrivalAt_angle_deg",  # redundant with decimal_hour
            #"legs1_arrivalAt_angle_deg",  # redundant with decimal_hour  
            #"legs0_departureAt_angle_deg",  # redundant with decimal_hour
            #"legs1_departureAt_angle_deg",  # redundant with decimal_hour
            #"log_route_frequency",  # route_frequency is more interpretable
        ])
        pa.default_memory_pool().release_unused()
        gc.collect()

    
    def add_price_rank_features(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("totalPrice").rank("dense").over("ranker_id").alias("price_rank"),
            pl.len().over("ranker_id").alias("group_count"),
            pl.col("legs0_duration").rank("dense").over("ranker_id").alias("duration_rank"),
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("price_rank") - 1) / (pl.col("group_count") - 1).cast(pl.Float64)).alias("price_pct_rank")
        ])
        self.trainDFPl = self.trainDFPl.drop("group_count")
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") + 1).log().alias("log_price")
        ])
        print("Done add_price_rank_features!!!")
        

    def add_min_stopage_per_trip(self): 
        # Reusing your logic, renamed to snake_case to be consistent
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")
        codeCols = [
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.sum_horizontal(pl.col(col).is_not_null().cast(pl.UInt8) for col in codeCols).alias("seg_tempor"), 
            pl.col("corporateTariffCode").is_not_null().cast(pl.Int32).alias("has_corporate_tariff")
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("seg_tempor") == pl.col("seg_tempor").min().over("ranker_id")).cast(pl.Int32).alias("minStopagePerRankderID")
        ])
        self.trainDFPl = self.trainDFPl.drop("seg_tempor")

        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("miniRules1_monetaryAmount") == 0) & (pl.col("miniRules1_statusInfos") == 1)).cast(pl.Int8).alias("free_exchange")
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            freq_split.alias("freq_split"),  
            pl.sum_horizontal([pl.col(col).is_in(freq_split).cast(pl.Int32)for col in codeCols]).alias("frequentFlyer_marketingCarrier_match")
        ])
        self.trainDFPl = self.trainDFPl.drop("freq_split")

        del codeCols, freq_split
        pa.default_memory_pool().release_unused()
        gc.collect()
        
        print("Done add_min_stopage_per_trip!!!")

    def ff_flyer_bin_converter(self): 
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")

        codeCols = [
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(freq_split.alias("freq_split"))
        
        matches_exprs = [
            pl.when(pl.col(col).is_in(pl.col("freq_split"))).then(1).otherwise(0).alias(f"match_{col}")
            for col in codeCols
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(matches_exprs)
        
        match_cols = [f"match_{col}" for col in codeCols]
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(match_cols).alias("numberof_same_frequentFlyter_operator"),
            pl.col("bySelf").cast(pl.Int8).alias("bySelf")
        )
        
        drop_cols = codeCols + ["frequentFlyer"] + match_cols + ["freq_split"]
        self.trainDFPl = self.trainDFPl.drop(drop_cols)
        
        del codeCols, freq_split, matches_exprs, match_cols, drop_cols
        pa.default_memory_pool().release_unused()
        gc.collect()

        # Convert binary features to int8 safely
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("isAccess3D").fill_null(False).cast(pl.Int8),
            pl.col("isVip").fill_null(False).cast(pl.Int8),
            pl.col("sex").fill_null(False).cast(pl.Int8),
            pl.col("has_corporate_tariff").fill_null(False).cast(pl.Int8),
        ])
        
        # Aircraft code presence binary
        aircraft_cols = [
            "legs0_segments0_aircraft_code",
            "legs0_segments1_aircraft_code",
            "legs0_segments2_aircraft_code",
            "legs0_segments3_aircraft_code",
            "legs1_segments0_aircraft_code",
            "legs1_segments1_aircraft_code",
            "legs1_segments2_aircraft_code",
            "legs1_segments3_aircraft_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns(
            [pl.col(c).is_not_null().cast(pl.Int8).alias(c) for c in aircraft_cols]
        )
        
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(aircraft_cols).alias("total_travel_stopage")
        )
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.col("total_travel_stopage").min().over("ranker_id").alias("minimum_travel_segment"),
        )
        self.trainDFPl = self.trainDFPl.drop(aircraft_cols)
        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done ff_flyer_bin_converter!")

    def hhmmss_to_minutes(self, col_name) -> pl.Expr:
        def parse_duration(x):
            if len(x) < 3:
                return None
            first_part = x[0]
            minutes = float(x[1])
            seconds = float(x[2]) if len(x) > 2 else 0
            if '.' in first_part:
                day_str, hour_str = first_part.split('.')
                days = int(day_str)
                hours = int(hour_str)
            else:
                days = 0
                hours = int(first_part)
            total_minutes = days * 24 * 60 + hours * 60 + minutes + seconds / 60
            return total_minutes
    
        return (
            pl.col(col_name)
            .fill_null("00:00:00")
            .str.split(":")
            .map_elements(parse_duration, return_dtype=pl.Float64)
            .alias(col_name)
        )
    
    def hour_min_and_tax_converter(self): 
        self.trainDFPl = self.trainDFPl.with_columns([
            self.hhmmss_to_minutes("legs0_duration"),
            self.hhmmss_to_minutes("legs1_duration"),
        ])
        
        self.trainDFPl = self.trainDFPl.with_columns(
            (pl.col("legs0_duration") + pl.col("legs1_duration")).alias("total_travel_time_in_minutes")
        )
    
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("searchRoute").str.len_chars() // 3).alias("searchRoute"),
            (pl.col("taxes") / pl.col("totalPrice")).alias("tax_percentage"),
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("legs0_duration").rank("dense").over("ranker_id").alias("legs0_duration_rank"),
            pl.col("legs1_duration").rank("dense").over("ranker_id").alias("legs1_duration_rank"),
            pl.col("total_travel_time_in_minutes").rank("dense").over("ranker_id").alias("total_travel_time_in_minutes_duration_rank"),
        ])
        gc.collect()
        print("Done hour_min_and_tax_converter")

    def others_and_duration(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("taxes") / pl.col("pricingInfo_passengerCount")).alias("taxes"),
            (pl.col("totalPrice")/pl.col("pricingInfo_passengerCount")).alias("totalPrice")
        ])

        min_vals = (
            self.trainDFPl
            .group_by("ranker_id") 
            .agg([
                pl.col("totalPrice").min().alias("min_totalPrice"),
                pl.col("legs0_duration").min().alias("min_legs0_duration"),
                pl.col("legs1_duration").min().alias("min_legs1_duration"),
                pl.col("total_travel_time_in_minutes").min().alias("min_total_travel_time"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(min_vals, on="ranker_id", how="left")
        
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") - pl.col("min_totalPrice")).alias("totalPrice_diff"),
            (pl.col("legs0_duration") - pl.col("min_legs0_duration")).alias("legs0Duration_diff"),
            (pl.col("legs1_duration") - pl.col("min_legs1_duration")).alias("legs1Duration_diff"),
            (pl.col("total_travel_time_in_minutes") - pl.col("min_total_travel_time")).alias("totalDuration_diff"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "min_totalPrice",
            "min_legs0_duration",
            "min_legs1_duration",
            "min_total_travel_time", 
            "pricingInfo_passengerCount",
            "corporateTariffCode"
        ])

        self.trainDFPl = self.trainDFPl.with_columns(
            ((pl.col("totalPrice") + 1)/(pl.col("legs0_duration").fill_null(0) + pl.col("legs1_duration").fill_null(0) + 1)).alias("priceDuration_TradeOff")
        )        
        gc.collect()
        print("Done others_and_duration")

    def arrival_and_monetory_adder(self): 
        cols = ['legs0_arrivalAt', 'legs0_departureAt', 'legs1_arrivalAt', 'legs1_departureAt']
        for x in cols:
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(x).str.strip_chars().str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S", strict=False).alias(f"{x}_parsed")
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(f"{x}_parsed").dt.hour().fill_null(0).alias(f"{x}_hour"),
                pl.col(f"{x}_parsed").dt.minute().fill_null(0).alias(f"{x}_minute"),
                pl.col(f"{x}_parsed").dt.weekday().fill_null(0).alias(f"{x}_weekday"),  # added weekday here
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col(f"{x}_hour") + (pl.col(f"{x}_minute") / 60)).alias(f"{x}_decimal_hour")
            ])
        
            # REMOVED: angle_deg columns (these are now in the drop list)
            # self.trainDFPl = self.trainDFPl.with_columns([
            #     (((pl.col(f"{x}_hour") * 3600 + pl.col(f"{x}_minute") * 60) / 86400) * 360).alias(f"{x}_angle_deg")
            # ])

            # Add cyclical encoding for weekday (0=Mon, 6=Sun)
            self.trainDFPl = self.trainDFPl.with_columns([
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.sin, return_dtype=pl.Float64).alias(f"{x}_weekday_sin"),
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.cos, return_dtype=pl.Float64).alias(f"{x}_weekday_cos"),
            ])
            
            #  Red-eye indicator (late night/early morning)
            self.trainDFPl = self.trainDFPl.with_columns([
                ((self.trainDFPl[f"{x}_hour"] >= 23) | (self.trainDFPl[f"{x}_hour"] < 6)).cast(pl.Int32).alias(f"{x}_is_redeye")
            ])

            # Drop intermediates
            self.trainDFPl = self.trainDFPl.drop([f"{x}_parsed", f"{x}_hour", f"{x}_minute"])

        del cols
        gc.collect()

        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("miniRules0_monetaryAmount") / pl.col("totalPrice")).alias("miniRules0_monetaryAmount_ratio"),
            (pl.col("miniRules1_monetaryAmount") / pl.col("totalPrice")).alias("miniRules1_monetaryAmount_ratio"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "miniRules0_monetaryAmount",
            "miniRules1_monetaryAmount",
            "legs0_arrivalAt",
            "legs0_departureAt",
            "legs1_arrivalAt",
            "legs1_departureAt",
            "legs0_duration",
            "legs1_duration",
            "bySelf",
        ])
        gc.collect()
        print("Done arrival_and_monetory_adder")

    def add_advanced_ranking_features(self):
        """Add sophisticated ranking features that capture user preferences"""
        
        # Cross-group statistical features (very powerful for ranking)
        self.trainDFPl = self.trainDFPl.with_columns([
            # Price percentiles within ranker group
            pl.col("totalPrice").quantile(0.25).over("ranker_id").alias("price_q25"),
            pl.col("totalPrice").quantile(0.75).over("ranker_id").alias("price_q75"),
            pl.col("totalPrice").median().over("ranker_id").alias("price_median"),
            
            # Duration percentiles
            pl.col("total_travel_time_in_minutes").quantile(0.25).over("ranker_id").alias("duration_q25"),
            pl.col("total_travel_time_in_minutes").quantile(0.75).over("ranker_id").alias("duration_q75"),
            pl.col("total_travel_time_in_minutes").median().over("ranker_id").alias("duration_median"),
        ])
        
        # Relative position features (crucial for ranking)
        self.trainDFPl = self.trainDFPl.with_columns([
            # Where this option stands relative to quartiles
            ((pl.col("totalPrice") - pl.col("price_q25")) / (pl.col("price_q75") - pl.col("price_q25") + 1e-8)).alias("price_quartile_position"),
            ((pl.col("total_travel_time_in_minutes") - pl.col("duration_q25")) / (pl.col("duration_q75") - pl.col("duration_q25") + 1e-8)).alias("duration_quartile_position"),
            
            # Distance from median (users often prefer median options)
            (pl.col("totalPrice") - pl.col("price_median")).abs().alias("price_deviation_from_median"),
            (pl.col("total_travel_time_in_minutes") - pl.col("duration_median")).abs().alias("duration_deviation_from_median"),
        ])
        
        # Clean up intermediate columns
        self.trainDFPl = self.trainDFPl.drop(["price_q25", "price_q75", "price_median", "duration_q25", "duration_q75", "duration_median"])

        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_advanced_ranking_features!!!")

    def add_user_company_features(self):
        """Add user and company preference patterns"""
        
        # Company-level aggregations (companies often have travel policies)
        company_stats = (
            self.trainDFPl.group_by("companyID")
            .agg([
                pl.col("totalPrice").mean().alias("company_avg_price"),
                pl.col("total_travel_time_in_minutes").mean().alias("company_avg_duration"),
                pl.col("legs0_segments0_cabinClass").mode().first().alias("company_preferred_cabin"),
                pl.col("isVip").mean().alias("company_vip_ratio"),
                pl.col("free_exchange").mean().alias("company_free_exchange_ratio"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(company_stats, on="companyID", how="left")
        
        # How this flight compares to company preferences
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") / (pl.col("company_avg_price") + 1)).alias("price_vs_company_avg"),
            (pl.col("total_travel_time_in_minutes") / (pl.col("company_avg_duration") + 1)).alias("duration_vs_company_avg"),
            (pl.col("legs0_segments0_cabinClass") == pl.col("company_preferred_cabin")).cast(pl.Int8).alias("matches_company_cabin_pref"),
        ])
        
        # Nationality preferences (cultural travel patterns)
        nationality_stats = (
            self.trainDFPl.group_by("nationality")
            .agg([
                pl.col("totalPrice").mean().alias("nationality_avg_price"),
                pl.col("legs0_segments0_cabinClass").mode().first().alias("nationality_preferred_cabin"),
                pl.col("minStopagePerRankderID").mean().alias("nationality_prefers_direct"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(nationality_stats, on="nationality", how="left")
        
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") / (pl.col("nationality_avg_price") + 1)).alias("price_vs_nationality_avg"),
            (pl.col("legs0_segments0_cabinClass") == pl.col("nationality_preferred_cabin")).cast(pl.Int8).alias("matches_nationality_cabin_pref"),
        ])
        
        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_user_company_features!!!")

    def add_advanced_time_features(self):
        """Add sophisticated time-based features"""
        
        # Time preference patterns within each ranker group
        time_cols = [
            "legs0_departureAt_decimal_hour", "legs0_arrivalAt_decimal_hour",
            "legs1_departureAt_decimal_hour", "legs1_arrivalAt_decimal_hour"
        ]
        
        for col in time_cols:
            if col in self.trainDFPl.columns:
                # Time clustering (morning, afternoon, evening, night)
                self.trainDFPl = self.trainDFPl.with_columns([
                    pl.when(pl.col(col) < 6).then(0)  # night
                    .when(pl.col(col) < 12).then(1)   # morning  
                    .when(pl.col(col) < 18).then(2)   # afternoon
                    .otherwise(3).alias(f"{col}_time_cluster"),
                    
                    # Peak vs off-peak (business travel patterns)
                    ((pl.col(col) >= 7) & (pl.col(col) <= 9) | 
                     (pl.col(col) >= 17) & (pl.col(col) <= 19)).cast(pl.Int8).alias(f"{col}_is_peak_time"),
                ])
        
        # Weekend vs weekday patterns (using your weekday columns if they exist)
        weekday_cols = [col for col in self.trainDFPl.columns if "_weekday_sin" in col or "_weekday_cos" in col]
        if len(weekday_cols) >= 2:
            # Is weekend flight (Friday evening to Sunday)
            self.trainDFPl = self.trainDFPl.with_columns([
                ((pl.col("legs0_departureAt_weekday_cos") > 0.5) |  # Friday-Sunday pattern
                 (pl.col("legs0_departureAt_weekday_sin") < -0.5)).cast(pl.Int8).alias("is_weekend_departure")
            ])

        # Cleaning up some intermediate columns that might cause issues
        cleanup_cols = [
            "legs0_arrivalAt_weekday_cos", "legs0_departureAt_weekday_sin", "legs0_arrivalAt_weekday_sin",
            "legs1_departureAt_weekday_sin", "legs1_arrivalAt_weekday_sin", "legs0_departureAt_weekday_cos"
        ]
        existing_cleanup_cols = [col for col in cleanup_cols if col in self.trainDFPl.columns]
        if existing_cleanup_cols:
            self.trainDFPl = self.trainDFPl.drop(existing_cleanup_cols)
            
        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_advanced_time_features!!!")

    def add_flight_quality_features(self):
        """Add features that capture flight quality and convenience"""
        
        # Baggage generosity score
        baggage_cols = [col for col in self.trainDFPl.columns if "baggageAllowance_quantity" in col]
        if baggage_cols:
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.sum_horizontal([pl.col(col).fill_null(0) for col in baggage_cols]).alias("total_baggage_allowance")
            ])
            
            # Baggage rank within group
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col("total_baggage_allowance").rank("dense").over("ranker_id").alias("baggage_rank")
            ])
        
        # Seat availability score (more seats = more flexible)
        seat_cols = [col for col in self.trainDFPl.columns if "seatsAvailable" in col]
        if seat_cols:
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.sum_horizontal([pl.col(col).fill_null(0) for col in seat_cols]).alias("total_seats_available")
            ])
            
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col("total_seats_available").rank("dense").over("ranker_id").alias("seat_availability_rank")
            ])
        
        # Cabin class consistency (same class throughout journey)
        cabin_cols = [col for col in self.trainDFPl.columns if "cabinClass" in col]
        if len(cabin_cols) >= 2:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col(cabin_cols[0]) == pl.col(cabin_cols[1])).cast(pl.Int8).alias("consistent_cabin_class")
            ])
            
            # Premium cabin indicator
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.max_horizontal([pl.col(col).fill_null(1) for col in cabin_cols]).alias("max_cabin_class")
            ])
            
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("max_cabin_class") >= 2.0).cast(pl.Int8).alias("has_premium_cabin")
            ])

        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_flight_quality_features!!!")

    def add_competitive_features(self):
        """Add features showing how this option competes with others"""
        
        # Best-in-class indicators
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") == pl.col("totalPrice").min().over("ranker_id")).cast(pl.Int8).alias("is_cheapest"),
            (pl.col("total_travel_time_in_minutes") == pl.col("total_travel_time_in_minutes").min().over("ranker_id")).cast(pl.Int8).alias("is_fastest"),
            (pl.col("minStopagePerRankderID") == 1).cast(pl.Int8).alias("is_most_direct"),
        ])
        
        # Pareto efficiency (good on multiple dimensions)
        self.trainDFPl = self.trainDFPl.with_columns([
            # Score combining price and time (lower is better for both)
            ((1 / (pl.col("price_rank") + 1)) + (1 / (pl.col("duration_rank") + 1))).alias("pareto_score"),
            
            # Multi-objective rank
            (pl.col("price_rank") + pl.col("duration_rank")).alias("combined_rank")
        ])
        
        # Competition intensity
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.len().over("ranker_id").alias("num_options"),  # More options = more competition
            
            # Price spread in group
            (pl.col("totalPrice").max().over("ranker_id") - pl.col("totalPrice").min().over("ranker_id")).alias("price_range_in_group"),
        ])
        
        # Market share features (for routes/companies)
        route_stats = (
            self.trainDFPl.group_by("searchRoute")
            .agg([
                pl.len().alias("route_frequency"),
                pl.col("totalPrice").mean().alias("route_avg_price"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(route_stats, on="searchRoute", how="left")
        
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") / (pl.col("route_avg_price") + 1)).alias("price_vs_route_avg"),
            # REMOVED: log_route_frequency (now in drop list - use route_frequency directly)
        ])

        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_competitive_features!!!")

    def add_interaction_features(self):
        """Add powerful interaction features"""
        
        # Basic interactions first (from your original method)
        if "isVip" in self.trainDFPl.columns and "free_exchange" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("isVip") * pl.col("free_exchange")).alias("vip_free_exchange_interaction")
            ])
        
        # VIP and business interactions
        if "has_premium_cabin" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("isVip") * pl.col("has_premium_cabin")).alias("vip_premium_interaction"),
            ])
            
        if "is_cheapest" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("isVip") * pl.col("is_cheapest")).alias("vip_cheapest_interaction"),
            ])
            
        if "pricingInfo_isAccessTP" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("has_corporate_tariff") * pl.col("pricingInfo_isAccessTP")).alias("corporate_policy_interaction"),
            ])
        
        # Time and user type interactions
        if "is_weekend_departure" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("is_weekend_departure") * pl.col("isVip")).alias("weekend_vip_interaction"),
                (pl.col("is_weekend_departure") * pl.col("has_corporate_tariff")).alias("weekend_business_interaction"),
            ])
        
        # Price sensitivity interactions
        if "price_quartile_position" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("price_quartile_position") * pl.col("isVip")).alias("price_sensitivity_vip"),
                (pl.col("price_quartile_position") * pl.col("has_corporate_tariff")).alias("price_sensitivity_corporate"),
            ])

        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done add_interaction_features!!!")

    def returnProcessedDF(self):
        self.add_price_rank_features()
        self.add_min_stopage_per_trip()
        self.ff_flyer_bin_converter()
        self.hour_min_and_tax_converter()
        self.others_and_duration()
        self.arrival_and_monetory_adder()
        
        # Add all the advanced features
        self.add_advanced_ranking_features()
        self.add_user_company_features()
        self.add_advanced_time_features()
        self.add_flight_quality_features()
        self.add_competitive_features()
        self.add_interaction_features()  # This now includes both basic and advanced interactions
        
        return self.trainDFPl

In [5]:
trainDFPl = PLDFPorcessor(trainPath).returnProcessedDF().sort("ranker_id")
trainDFPl.head()

Done add_price_rank_features!!!
Done add_min_stopage_per_trip!!!
Done ff_flyer_bin_converter!
Done hour_min_and_tax_converter
Done others_and_duration
Done arrival_and_monetory_adder
Done add_advanced_ranking_features!!!
Done add_user_company_features!!!
Done add_advanced_time_features!!!
Done add_flight_quality_features!!!
Done add_competitive_features!!!
Done add_interaction_features!!!


companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs0_segments1_baggageAllowance_quantity,legs0_segments1_baggageAllowance_weightMeasurementType,legs0_segments1_cabinClass,legs0_segments1_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,legs1_segments1_baggageAllowance_quantity,legs1_segments1_baggageAllowance_weightMeasurementType,legs1_segments1_cabinClass,legs1_segments1_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,…,nationality_prefers_direct,price_vs_nationality_avg,matches_nationality_cabin_pref,legs0_departureAt_decimal_hour_time_cluster,legs0_departureAt_decimal_hour_is_peak_time,legs0_arrivalAt_decimal_hour_time_cluster,legs0_arrivalAt_decimal_hour_is_peak_time,legs1_departureAt_decimal_hour_time_cluster,legs1_departureAt_decimal_hour_is_peak_time,legs1_arrivalAt_decimal_hour_time_cluster,legs1_arrivalAt_decimal_hour_is_peak_time,is_weekend_departure,total_baggage_allowance,baggage_rank,total_seats_available,seat_availability_rank,consistent_cabin_class,max_cabin_class,has_premium_cabin,is_cheapest,is_fastest,is_most_direct,pareto_score,combined_rank,num_options,price_range_in_group,route_frequency,route_avg_price,price_vs_route_avg,vip_free_exchange_interaction,vip_premium_interaction,vip_cheapest_interaction,corporate_policy_interaction,weekend_vip_interaction,weekend_business_interaction,price_sensitivity_vip,price_sensitivity_corporate
i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u32,i8,f64,f64,i64,u32,u32,f64,f64,i8,i32,i8,i32,…,f64,f64,i8,i32,i8,i32,i8,i32,i8,i32,i8,i8,f64,u32,f64,u32,i8,f64,i8,i8,i8,i8,f64,u32,u32,f64,u32,f64,f64,i8,i8,i8,f64,i8,i8,f64,f64
41022,36,0,0,0.0,0.0,1.0,6.0,0.0,0.0,1.0,9.0,null,null,null,null,null,null,null,null,1.0,0.0,1.0,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,9040.0,0,1,5,0.0,9.109525,0,0,0,0,…,0.811694,0.19797,1,3,0,3,0,0,0,0,0,0,0.0,1,15.0,3,1,1.0,0,1,0,0,0.666667,6,15,22747.0,4387191,32801.871663,0.275586,0,0,0,0.0,0,0,-0.0,-0.0
41022,36,0,0,1.0,0.0,1.0,6.0,1.0,0.0,1.0,9.0,null,null,null,null,null,null,null,null,1.0,0.0,1.0,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,13740.0,0,3,5,0.142857,9.528139,0,0,0,0,…,0.811694,0.300896,1,3,0,3,0,0,0,0,0,0,2.0,3,15.0,3,1,1.0,0,0,0,0,0.416667,8,15,22747.0,4387191,32801.871663,0.418866,0,0,0,0.0,0,0,0.0,0.0
41022,36,0,0,1.0,0.0,1.0,6.0,1.0,0.0,1.0,9.0,null,null,null,null,null,null,null,null,1.0,1.0,1.0,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,20440.0,0,6,5,0.357143,9.925298,0,0,0,0,…,0.811694,0.447621,1,3,0,3,0,0,0,0,0,0,2.0,3,15.0,3,1,1.0,0,0,0,0,0.309524,11,15,22747.0,4387191,32801.871663,0.623116,0,0,0,0.0,0,0,0.0,0.0
41022,36,1,0,0.0,0.0,1.0,1.0,null,null,null,null,null,null,null,null,null,null,null,null,1.0,0.0,1.0,"""00004b95a9e446b586c5e9b7297144…",2,0,577.0,21897.0,0,7,1,0.428571,9.994151,1,1,0,0,…,0.811694,0.479529,1,0,0,0,0,0,0,0,0,0,0.0,1,1.0,1,null,1.0,0,0,1,1,0.625,8,15,22747.0,4387191,32801.871663,0.667533,0,0,0,1.0,0,0,0.0,1.217463
41022,36,1,0,1.0,0.0,1.0,1.0,null,null,null,null,null,null,null,null,null,null,null,null,1.0,1.0,1.0,"""00004b95a9e446b586c5e9b7297144…",2,0,577.0,24057.0,1,8,1,0.5,10.088223,1,1,0,0,…,0.811694,0.526831,1,0,0,0,0,0,0,0,0,0,1.0,2,1.0,1,null,1.0,0,0,1,1,0.611111,9,15,22747.0,4387191,32801.871663,0.733381,0,0,0,1.0,0,0,0.0,1.539851


In [6]:
for col_name in trainDFPl.columns:
    total_missing = (trainDFPl[col_name].is_null().sum()/len(trainDFPl[col_name]))*100
    print(f"{col_name}: {total_missing}")

gc.collect()

companyID: 0.0
nationality: 0.0
isAccess3D: 0.0
isVip: 0.0
legs0_segments0_baggageAllowance_quantity: 0.005863754129703155
legs0_segments0_baggageAllowance_weightMeasurementType: 0.005863754129703155
legs0_segments0_cabinClass: 0.0
legs0_segments0_seatsAvailable: 0.4397099161152497
legs0_segments1_baggageAllowance_quantity: 79.0648105753908
legs0_segments1_baggageAllowance_weightMeasurementType: 79.0648105753908
legs0_segments1_cabinClass: 79.05910113058029
legs0_segments1_seatsAvailable: 79.08736178018285
legs1_segments0_baggageAllowance_quantity: 24.950147067803293
legs1_segments0_baggageAllowance_weightMeasurementType: 24.950147067803293
legs1_segments0_cabinClass: 24.937559836194044
legs1_segments0_seatsAvailable: 25.24552817103998
legs1_segments1_baggageAllowance_quantity: 84.55122882021928
legs1_segments1_baggageAllowance_weightMeasurementType: 84.55122882021928
legs1_segments1_cabinClass: 84.53864158861003
legs1_segments1_seatsAvailable: 84.55793576455748
miniRules0_statusInfos:

0

In [7]:
#trainDFPl = trainDFPl.with_columns(
#    pl.col("legs0_segments0_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs0_segments0_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs0_segments0_seatsAvailable").fill_null(-1) , #.fill_nan(0)
#    
#    pl.col("legs0_segments1_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs0_segments1_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs0_segments1_seatsAvailable").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs0_segments1_cabinClass").fill_null(-1) , #.fill_nan(0)
#    
#    pl.col("legs1_segments0_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments0_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments0_seatsAvailable").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments0_cabinClass").fill_null(-1) ,
#    
#    pl.col("legs1_segments1_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments1_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments1_seatsAvailable").fill_null(-1) , #.fill_nan(0)
#    pl.col("legs1_segments1_cabinClass").fill_null(-1) ,
#    
#    pl.col("miniRules0_statusInfos").fill_null(-1) ,
#    pl.col("miniRules1_statusInfos").fill_null(-1) ,
#    pl.col("pricingInfo_isAccessTP").fill_null(-1) ,
#    pl.col("free_exchange").fill_null(-1) ,
#    pl.col("miniRules0_monetaryAmount_ratio").fill_null(-1) ,
#    pl.col("miniRules1_monetaryAmount_ratio").fill_null(-1) ,
#    pl.col("vip_free_exchange_interaction").fill_null(-1) ,
#)

#trainDFPl.head(15)

In [10]:
#for col_name in trainDFPl.columns:
#    total_missing = (trainDFPl[col_name].is_null().sum()/len(trainDFPl[col_name]))*100
#    print(f"{col_name}: {total_missing}")
#
gc.collect()

0

## ======== Now split the dataset ==========

In [7]:
#ranker_ids = trainDFPl["ranker_id"].unique().to_numpy()

# split DF based on ranker_id
#gss = GroupShuffleSplit(n_splits=1, test_size=0.01, random_state=42)
#train_idx, valid_idx = next(gss.split(ranker_ids, groups=ranker_ids))

#train_feature_cols = [col for col in trainDFPl.columns if col not in ("selected", "ranker_id")]

#train_rankers = ranker_ids[train_idx]
#valid_rankers = ranker_ids[valid_idx]

# train_df = trainDFPl.sort("ranker_id")
#train_df = trainDFPl.filter(pl.col("ranker_id").is_in(train_rankers)).sort("ranker_id")
#valid_df = trainDFPl.filter(pl.col("ranker_id").is_in(valid_rankers)).sort("ranker_id")

gc.collect()

0

In [8]:
#validDFPl = trainDFPl.filter(pl.col("ranker_id").is_in(valid_rankers))
#trainDFPl = trainDFPl.filter(pl.col("ranker_id").is_in(train_rankers))


gc.collect()
#del trainDFPl gss, train_idx, valid_idx, train_rankers, valid_rankers, ranker_ids
gc.collect()
pa.default_memory_pool().release_unused()
gc.collect()


0

# Create groups and prepare model trainer for xgb ranker: ===================

In [9]:
group_train_xgb = (
    trainDFPl.group_by("ranker_id").len()
    .sort("ranker_id")["len"]
    .to_numpy()
    .astype("uint32")
)

pa.default_memory_pool().release_unused()
gc.collect()

0

In [11]:
# create groups for XGB ranker and LGB ranker
#group_train_xgb = (
#    train_df.group_by("ranker_id").len()
#    .sort("ranker_id")["len"]
#    .to_numpy()
#    .astype("uint32")
#)
#group_valid_xgb = (
#    valid_df.group_by("ranker_id").len()
#    .sort("ranker_id")["len"]
#    .to_numpy()
#    .astype("uint32")
#)

pa.default_memory_pool().release_unused()
gc.collect()

0

In [10]:
trainDFPl_select = trainDFPl["selected"].to_numpy()
train_df = trainDFPl.drop(["selected", "ranker_id"]).to_numpy()

trainDFPl = None 
del trainDFPl
pa.default_memory_pool().release_unused()
gc.collect()

0

In [11]:
#train_df_select = train_df["selected"].to_numpy()
#train_df = train_df.drop(["selected", "ranker_id"]).to_numpy()

#valid_df_select = valid_df["selected"].to_numpy()
#valid_df = valid_df.drop(["selected", "ranker_id"]).to_numpy()

pa.default_memory_pool().release_unused()
gc.collect()

0

In [ ]:
dtrain_xgb = xgb.QuantileDMatrix(
    trainDFPl,
    label=trainDFPl_select,
    group=group_train_xgb
)

train_df = None
del train_df
pa.default_memory_pool().release_unused()
gc.collect()

In [ ]:
#dtrain_xgb = xgb.QuantileDMatrix(
#    train_df,
#    label=train_df_select,
#    group=group_train_xgb
#)

#train_df = None
#del train_df
#pa.default_memory_pool().release_unused()
#gc.collect()


#dvalid_xgb = xgb.QuantileDMatrix(
#    valid_df,
#    label=valid_df_select,
#    group=group_valid_xgb
#)
#valid_df = None 
#del valid_df
pa.default_memory_pool().release_unused()
gc.collect()

# DMatrix preparation for train
dtrain_xgb = xgb.DMatrix(   
    train_df , 
    label = train_df_select
) 
dtrain_xgb.set_group(group_train_xgb) 
 
train_df = None 
del train_df
pa.default_memory_pool().release_unused() 
gc.collect() 
 

# DMatrix preparation for valid
dvalid_xgb = xgb.DMatrix( 
    valid_df,  
    label = valid_df_select 
) 
dvalid_xgb.set_group(group_valid_xgb) 

valid_df = None  
del valid_df 
pa.default_memory_pool().release_unused()
gc.collect() 

### train xgb ranker model

In [ ]:
params = {
    'objective': 'rank:ndcg',
    "eval_metric": ["ndcg@3", "ndcg@10"],
    "learning_rate": 0.022641389657079056,
    "max_depth": 0,
    "min_child_weight": 2 , # 2
    "subsample": 0.8842234913702768,
    "colsample_bytree": 0.45840689146263086,
    "gamma": 3.3084297630544888,
    "lambda": 6.952586917313028,
    "alpha": 0.6395254133055179,
    'seed': 42,
    'n_jobs': -1,
    'device': 'cuda'
}


trainedXGBoostsModel = xgb.train(
    params = params,
    dtrain = dtrain_xgb,
    num_boost_round = 500,
    evals=[(dtrain_xgb, "train")], # [(dtrain_xgb, "train"), (dvalid_xgb, "valid")],
    # early_stopping_rounds=100,
    verbose_eval=10
)

In [ ]:
dtrain_xgb, dvalid_xgb, group_train_xgb, group_valid_xgb = None , None, None , None
del dtrain_xgb, dvalid_xgb, group_train_xgb, group_valid_xgb

pa.default_memory_pool().release_unused()
gc.collect()

## XGB Ranker: check the important cols according to their weights

In [ ]:
gain_importance = trainedXGBoostsModel.get_score(importance_type='gain')
colRename_map = {f"f{idx}": col for idx, col in enumerate(train_feature_cols)}

gain_importance_df = pd.DataFrame(
    list(gain_importance.items()),
    columns=['feature', 'importance']
)
gain_importance_df['feature'] = gain_importance_df['feature'].map(colRename_map)
gain_importance_df = gain_importance_df.sort_values(by='importance', ascending=False)
gain_importance_df.reset_index(drop=True, inplace=True)

print(len(gain_importance_df))
gain_importance_df

In [ ]:
gain_importance, colRename_map , gain_importance_df = None , None , None 
del gain_importance, colRename_map , gain_importance_df
pa.default_memory_pool().release_unused()
gc.collect()

# ==================== Predict and Dataset task ====================

# Now work for test Dataset for xgb ranker: --------------------

In [ ]:
testDFPl = PLDFPorcessor(testPath).returnProcessedDF().sort("ranker_id")

testIds_xgb = testDFPl["Id"].to_numpy()
testRanker_ids_xgb = testDFPl["ranker_id"].to_numpy()

test_feature_cols = [col for col in testDFPl.columns if col not in ("Id", "ranker_id")]

In [ ]:
passToPredict_df = testDFPl.select(test_feature_cols).to_numpy()

group_test = (
    testDFPl.group_by("ranker_id").len()
    .sort("ranker_id")["len"]
    .to_numpy()
    .astype("uint32")
)

pa.default_memory_pool().release_unused()
gc.collect()

In [ ]:
dtest = xgb.DMatrix(passToPredict_df)
dtest.set_group(group_test)

pred_xgbRanker = trainedXGBoostsModel.predict(dtest)

dtest, passToPredict_df, group_test = None, None, None
del dtest, passToPredict_df, group_test
pa.default_memory_pool().release_unused()
gc.collect()

## create a csv file for xgb ranker

In [ ]:
xgbSubmissionDF = pl.DataFrame({
    "Id": testIds_xgb,
    "ranker_id": testRanker_ids_xgb,
    "selected_xgb": pred_xgbRanker
})

xgbSubmissionDF = xgbSubmissionDF.with_columns([
    pl.col("selected_xgb")
    .rank(method="ordinal", descending=True)
    .over("ranker_id")
    .alias("rank_xgb")
])


xgbSubmissionDF = xgbSubmissionDF.sort("Id", descending=False) # descending=False -> small to big
xgbSubmissionDF.head(10)

In [ ]:
xgbSubmissionDF = xgbSubmissionDF.drop([
    "selected_xgb"
]).rename({"rank_xgb": "selected"})

In [ ]:
xgbSubmissionDF.head(10)

In [ ]:
xgbSubmissionDF.write_csv(os.path.join(os.path.abspath("."), "submission.csv"))